# YgoMaster-PackBuilder — Manifest Validation Tests

Notebook di test manuale/esplorativo per il **manifest validation layer**.

Obiettivi:

- caricare il manifest reale;
- verificare che il manifest corrente sia valido;
- controllare `ValidationReport`;
- testare una regola di validation alla volta usando copie modificate in memoria;
- verificare che gli errori siano raccolti senza interrompere la validation al primo problema.

## 1. Ambiente e root della repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository. "
        "Esegui il notebook dalla root del progetto oppure dalla cartella tests/."
    )

manifest_path = repo_root / "config" / "project.yaml"

assert manifest_path.exists(), f"Manifest non trovato: {manifest_path}"

print("Repository root:", repo_root)
print("Manifest path:", manifest_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Manifest path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\project.yaml


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.manifest import load_project_manifest

from ygomaster_packbuilder.validation import (
    ValidationIssue,
    ValidationReport,
    ValidationSeverity,
    validate_project_manifest,
)

print("Validation imports OK")

Validation imports OK


## 3. Helper per visualizzare un report

In [3]:
def show_report(report: ValidationReport) -> None:
    print("Valid:", report.is_valid)
    print("Errors:", len(report.errors))
    print("Warnings:", len(report.warnings))
    print()

    for issue in report.issues:
        print(
            f"[{issue.severity.value.upper()}] "
            f"{issue.code} @ {issue.path}"
        )
        print(" ", issue.message)
        print()


def issue_codes(report: ValidationReport) -> list[str]:
    return [issue.code for issue in report.issues]


def assert_has_code(report: ValidationReport, code: str) -> None:
    assert code in issue_codes(report), (
        f"Expected validation code {code!r}, "
        f"found {issue_codes(report)!r}"
    )

print("Helpers ready")

Helpers ready


## 4. Caricamento del manifest reale

In [4]:
manifest = load_project_manifest(manifest_path)

print("Loaded:", manifest.settings.name)

Loaded: Master Duel Custom League


## 5. Manifest reale: validation

Il manifest corrente dovrebbe essere valido secondo le regole implementate finora.


In [5]:
report = validate_project_manifest(manifest)

show_report(report)

assert isinstance(report, ValidationReport)
assert report.is_valid
assert report.errors == []

print("Current manifest is valid")

Valid: True
Errors: 0
Warnings: 0

Current manifest is valid


## 6. `ValidationReport`: proprietà base

In [6]:
empty_report = ValidationReport()

assert empty_report.is_valid
assert empty_report.errors == []
assert empty_report.warnings == []

test_report = ValidationReport(
    issues=[
        ValidationIssue(
            code="example-warning",
            message="Example warning",
            path="example",
            severity=ValidationSeverity.WARNING,
        ),
        ValidationIssue(
            code="example-error",
            message="Example error",
            path="example",
            severity=ValidationSeverity.ERROR,
        ),
    ]
)

assert not test_report.is_valid
assert len(test_report.errors) == 1
assert len(test_report.warnings) == 1

show_report(test_report)

Valid: False
Errors: 1
Warnings: 1

[WARNING] example-warning @ example
  Example warning

[ERROR] example-error @ example
  Example error



## 7. Nessun player

Regola attesa:

```text
no-players
```


In [7]:
case = deepcopy(manifest)
case.players.clear()

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "no-players")

Valid: False
Errors: 1
Warnings: 0

[ERROR] no-players @ players
  The manifest must define at least one player.



## 8. Player key duplicata

Regola attesa:

```text
duplicate-player-key
```


In [8]:
case = deepcopy(manifest)

assert len(case.players) >= 1

duplicate = deepcopy(case.players[0])
case.players.append(duplicate)

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "duplicate-player-key")

Valid: False
Errors: 5
Warnings: 0

[ERROR] duplicate-player-key @ players[3].key
  Player key 'livio' is defined more than once.

[ERROR] duplicate-pack-key @ players[3].main_pack.key
  Pack key 'livio-scaled_invasion' is defined more than once.

[ERROR] duplicate-pack-key @ players[3].mini_packs[0].key
  Pack key 'livio-scaled_invasion-molting_evolution' is defined more than once.

[ERROR] duplicate-pack-key @ players[3].mini_packs[1].key
  Pack key 'livio-scaled_invasion-venomous_strike' is defined more than once.

[ERROR] duplicate-pack-key @ players[3].mini_packs[2].key
  Pack key 'livio-scaled_invasion-alien_attack' is defined more than once.



## 9. Numero errato di mini-pack

Regola attesa:

```text
invalid-mini-pack-count
```


In [9]:
case = deepcopy(manifest)

player = case.players[0]

assert len(player.mini_packs) >= 1

player.mini_packs.pop()

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-mini-pack-count")

Valid: False
Errors: 1
Warnings: 0

[ERROR] invalid-mini-pack-count @ players[0].mini_packs
  Player 'livio' must define exactly 3 mini packs, but 2 were found.



## 10. Pack key duplicata

Le key dei pack devono essere uniche globalmente, anche tra categorie diverse.

Regola attesa:

```text
duplicate-pack-key
```


In [10]:
case = deepcopy(manifest)

player = case.players[0]
assert len(player.mini_packs) >= 1

player.mini_packs[0].key = player.main_pack.key

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "duplicate-pack-key")

Valid: False
Errors: 1
Warnings: 0

[ERROR] duplicate-pack-key @ players[0].mini_packs[0].key
  Pack key 'livio-scaled_invasion' is defined more than once.



## 11. Provider non supportato

Regola attesa:

```text
unsupported-source-provider
```


In [11]:
case = deepcopy(manifest)

case.players[0].main_pack.source.provider = "unsupported-provider"

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unsupported-source-provider")

Valid: False
Errors: 1
Warnings: 0

[ERROR] unsupported-source-provider @ players[0].main_pack.source.provider
  Source provider 'unsupported-provider' is not supported.



## 12. Shared pack type non supportato

Regola attesa:

```text
unsupported-shared-pack-type
```


In [12]:
case = deepcopy(manifest)

assert len(case.shared_packs) >= 1

case.shared_packs[0].type = "unknown_shared_type"

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "unsupported-shared-pack-type")

Valid: False
Errors: 2
Warnings: 0

[ERROR] unsupported-shared-pack-type @ shared_packs[0].type
  Shared pack type 'unknown_shared_type' is not supported.

[ERROR] invalid-shared-staples-count @ shared_packs
  There must be exactly one shared staples pack, found 0.



## 13. Numero errato di shared staples pack

Il progetto corrente richiede esattamente un `shared_staples`.

Regola attesa:

```text
invalid-shared-staples-count
```


In [13]:
case = deepcopy(manifest)

case.shared_packs.clear()

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-shared-staples-count")

Valid: False
Errors: 1
Warnings: 0

[ERROR] invalid-shared-staples-count @ shared_packs
  There must be exactly one shared staples pack, found 0.



## 14. Default opening policy vuota

Regola attesa:

```text
empty-default-opening-policy
```


In [14]:
case = deepcopy(manifest)

case.settings.default_opening_policy = ""

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "empty-default-opening-policy")

Valid: False
Errors: 1
Warnings: 0

[ERROR] empty-default-opening-policy @ settings.default_opening_policy
  Default opening policy must be 'configuration-a', found ''.



## 15. Override opening policy vuoto

Regola attesa:

```text
empty-opening-policy
```


In [15]:
case = deepcopy(manifest)

case.players[0].main_pack.opening_policy = ""

report = validate_project_manifest(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "empty-opening-policy")

Valid: False
Errors: 1
Warnings: 0

[ERROR] empty-opening-policy @ players[0].main_pack.opening_policy
  Opening policy override cannot be empty.



## 16. Raccolta di più errori contemporaneamente

Il validator non deve fermarsi al primo errore.


In [16]:
case = deepcopy(manifest)

case.settings.default_opening_policy = ""

player = case.players[0]

if player.mini_packs:
    player.mini_packs.pop()

player.main_pack.source.provider = "unsupported-provider"

if case.shared_packs:
    case.shared_packs[0].type = "invalid-shared-type"

report = validate_project_manifest(case)

show_report(report)

expected_codes = {
    "empty-default-opening-policy",
    "invalid-mini-pack-count",
    "unsupported-source-provider",
    "unsupported-shared-pack-type",
    "invalid-shared-staples-count",
}

found_codes = set(issue_codes(report))
missing = expected_codes - found_codes

assert not missing, (
    f"Expected validation issues not found: {missing}. "
    f"Found: {found_codes}"
)

assert len(report.errors) >= len(expected_codes)

print("Multiple-error collection OK")

Valid: False
Errors: 5
Warnings: 0

[ERROR] invalid-mini-pack-count @ players[0].mini_packs
  Player 'livio' must define exactly 3 mini packs, but 2 were found.

[ERROR] unsupported-source-provider @ players[0].main_pack.source.provider
  Source provider 'unsupported-provider' is not supported.

[ERROR] unsupported-shared-pack-type @ shared_packs[0].type
  Shared pack type 'invalid-shared-type' is not supported.

[ERROR] invalid-shared-staples-count @ shared_packs
  There must be exactly one shared staples pack, found 0.

[ERROR] empty-default-opening-policy @ settings.default_opening_policy
  Default opening policy must be 'configuration-a', found ''.

Multiple-error collection OK


## 17. Controllo dei path degli errori

I path devono indicare la posizione logica del dato errato.


In [17]:
case = deepcopy(manifest)

case.players[0].main_pack.source.provider = "bad-provider"

report = validate_project_manifest(case)

matching = [
    issue
    for issue in report.issues
    if issue.code == "unsupported-source-provider"
]

assert matching

issue = matching[0]

print("Code:", issue.code)
print("Path:", issue.path)
print("Message:", issue.message)

assert issue.path.endswith(".source.provider")

Code: unsupported-source-provider
Path: players[0].main_pack.source.provider
Message: Source provider 'bad-provider' is not supported.


## 18. Risultato

Se tutte le celle precedenti passano, il primo validation layer del manifest è operativo.

Abbiamo verificato:

```text
ProjectManifest
      ↓
validate_project_manifest(...)
      ↓
ValidationReport
      ├── errors
      ├── warnings
      └── is_valid
```

e le regole correnti:

- almeno un player;
- player key uniche;
- esattamente tre mini-pack per player;
- pack key globalmente uniche;
- provider supportati;
- shared pack type supportati;
- esattamente un shared staples pack;
- default opening policy non vuota;
- eventuali override policy non vuoti;
- raccolta di più errori nella stessa esecuzione.

Il controllo dell'esistenza reale delle opening policy verrà aggiunto quando sarà disponibile il relativo file/config loader.
